# Milestone 2 — LaTeX Processing

## Objectives

This milestone focuses on parsing LaTeX source files,
identifying document structure, and extracting relevant elements.

In [ ]:
# Setup: locate repo root so imports and paths work from any working directory
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

from src.utils.table_io import load_table, PROCESSED_DIR  # fallback loader

# Part II — Milestone 2: Hierarchical Parsing and Standardization

## 6. Parsing methodology

### 6.1 Main compilation path

The publication root is the `.tex` file that contains `\documentclass`. From this root the parser follows `\input` and `\include` commands transitively. Only the files reachable this way are parsed, so the parse covers what a reader of the compiled PDF sees. Every other `.tex` file in the source directory is treated as a redundant auxiliary file and ignored.

The implementation is designed to handle the following cases:

- **Root detection.** The root is the file containing `\documentclass`. When several files contain it, the tie-break rule is *TODO: e.g. the file that is not included by another file / the largest file*. When no file contains it, the version is recorded as `empty`.
- **Relative paths and omitted extensions.** Include targets are resolved relative to the source directory, and `.tex` is appended when the extension is omitted (`\input{sections/intro}` → `sections/intro.tex`).
- **Recursive includes and cycles.** The traversal keeps a set of visited files, so a file that includes itself, directly or indirectly, is read only once and cannot cause an infinite loop.
- **Comments.** Commands that are commented out with `%` are treated as plain text, not as includes. An escaped `\%` is not a comment.
- **Missing included files.** A target that cannot be found is logged, and the parser continues with the remaining files. *TODO: state whether the version is flagged as partial.*
- **Versions without TeX sources.** Milestone 1 keeps these as empty records. The parser records them with an `empty` status in `parse_manifest.csv` instead of skipping them, so coverage statistics reflect reality.

**Implementation:** *TODO: name the module and functions, e.g. `src/latexParser/...`, once the code is final.*

In [ ]:
# TODO: Link or import the actual parser implementation from src/latexParser/.
# Example:
# from src.latexParser import find_root_tex, collect_compilation_files, parse_publication

### 6.2 Hierarchy and element extraction

Each version of a publication is converted into a tree. The **document** is the root, sections or chapters form the second level, and subsections, paragraphs and other logical divisions form the lower levels. The nesting depends on each paper's formatting, so the parser detects the level of each sectioning command automatically instead of using a fixed order. *TODO: describe the detection rule, e.g. rank by the sectioning commands actually used in the paper.*

The leaves of the tree are:

- **Sentences** from body text.
- **Block formulas**, such as the content of an `equation` environment or of `$$...$$`.
- **Figures**. Tables are treated as a type of figure.

Itemized and enumerated lists are branching components: the `itemize` or `enumerate` block is a higher-level element, and each `\item` is an element at the next level.

Acknowledgements and appendices are included in the tree, including unnumbered `\section*` sections. The references section is not parsed into the hierarchy.

**Element kinds and hierarchy-level rules:** *TODO: list the exact `kind` values stored in `elements` (the statement's example uses `document`, `heading`, `sentence`) and the parent rule for each.*

**Sentence boundary handling:** A period that does not end a sentence must not create a new leaf. The splitter therefore does not split inside mathematics or LaTeX commands, after abbreviations such as `i.e.`, `e.g.`, `et al.` and `Fig. 1`, inside URLs, or inside decimal numbers. *TODO: describe the technique, e.g. masking protected spans before splitting.*

**Formula / figure / table handling:** Block formulas become formula leaves. Figure and table environments become figure leaves, with the LaTeX commands kept intact as Milestone 1 specified. *TODO: state what content is stored for a figure, e.g. the caption or the raw environment.*

### 6.3 Cleanup and math normalization

Formatting commands that carry no semantic meaning are removed so that text can be compared reliably across versions. The statement's examples are `\centering`, the `[htpb]` placement option and `\midrule`. Semantic content such as text, formulas, captions and citations is preserved.

**Cleanup rules:** *TODO: list the final set of removed commands and options.*

**Inline math format:** All inline mathematics is converted to the `$...$` form (`\(...\)` becomes `$...$`).

**Block math format:** All block mathematics is converted to `\begin{equation}...\end{equation}` (`$$...$$` and `\[...\]` are converted to it). *TODO: confirm how `align`, `eqnarray` and similar environments are handled.*

**Why this matters:** Cleanup and normalization run before content comparison. Without them, two copies of the same sentence that differ only in spacing or formatting would be seen as different elements and would not be deduplicated across versions.

**Validation checks:** *TODO: e.g. unit tests on small LaTeX snippets, plus a check that no `$$` or `\[` remains in `elements`.*

### 6.4 `\bibitem` conversion and reference deduplication

**Conversion method:** Each `\bibitem{key} ...` definition found in the `.tex` files is converted into a standard BibTeX entry with regular expressions. The citation key becomes the entry name, and the free-text body is split into fields such as authors, title, year, journal, doi and url. *TODO: state which fields are extracted, how `entry_type` is chosen, and what happens when a field cannot be recovered (left empty, not guessed).*

Deduplication works across the versions of one publication:

- **Matching / canonical-key policy.** Two entries are considered the same reference when their content refers to the same work, even if their citation keys differ. One citation key is then chosen as canonical. *TODO: state the matching rule (e.g. normalized title, doi) and the key-choice rule.*
- **Field union policy.** The unified entry takes the union of the fields of all duplicates and does not keep one entry while discarding the others. For example, if version 1 has `vaswani2017` with title, author and year, version 2 renames it to `vit_transformer` and adds `doi`, and version 3 keeps `vit_transformer` but drops `year`, the unified entry keeps one key and has title, author, year and doi. *TODO: state how conflicting values for the same field are resolved.*
- **Citation-key rewriting.** Every `\cite{}` command that uses a dropped key is renamed to the canonical key, so each cited key exists in the final `bibtex` table.

**Citation-key rewriting and tests:** *TODO: describe the tests, and note that the Citation integrity check in section 7.2 verifies the result on real output.*

### 6.5 Full-text content deduplication across versions

After cleanup, elements whose text matches exactly across versions are stored once and share one `element_id`. The `hierarchy` table still stores the node-parent edges separately for every version, so a shared element can have different parents in different versions.

- **Normalization before comparison.** Comparison is done on the cleaned and normalized text, so differences in formatting or whitespace do not hide identical content. *TODO: state the exact normalization, e.g. whitespace collapse.*
- **ID generation.** IDs start with the publication key followed by a discriminant that is unique inside the publication, for example `2310-12345#s1`. *TODO: describe the discriminant scheme.*
- **Identical content under different parents.** One element row serves all versions. Each version contributes its own `hierarchy` edge from that element to that version's parent, as in the statement's example where sentence `s1` hangs under `h1` in version 1 and under `h2` in version 2.
- **Root/document elements.** Each version has its own document root (`root1`, `root2`). Roots are not merged across versions. *TODO: confirm.*

**Implementation:** IDs follow `<pub_key>#<prefix><n>` (e.g. `2310-12345#s1`), where `pub_key` is the arXiv ID with `.` replaced by `-` and `n` is a per-publication counter, so IDs are unique inside a publication. After cleanup, whitespace is collapsed and elements are keyed by (kind, text); equal keys across versions reuse one `element_id`. Each version has one `document` root (`#root<version>`), never deduplicated and appearing only as a `parent_id`. `hierarchy` stores one edge per node per version.

## 7. Milestone 2 — output tables and validation

### 7.1 Expected tables

**`elements`** — one row per distinct cleaned element across all versions:
`element_id`, `arxiv_id`, `kind`, `content`

**`hierarchy`** — one row per node-parent edge for each version:
`arxiv_id`, `version`, `element_id`, `parent_id`

**`bibtex`** — unified BibTeX entries after deduplication and field merging:
`arxiv_id`, `citation_key`, `entry_type`, `title`, `authors`, `year`, `doi`, `journal`, `url`

### 7.2 Coverage and correctness checks

Before analysing the processed data, we verify that the output tables are complete and internally consistent. We run six checks:

- **Parsing success rate**: the share of paper versions that were parsed successfully, out of all versions (including empty and failed ones).
- **Version coverage**: how many versions each paper has, and how many of them have parsed content or are recorded as empty (no TeX source).
- **Deduplication effect**: the number of elements and BibTeX entries before and after deduplication.
- **Hierarchy integrity**: every `parent_id` in `hierarchy` exists in `elements`, and no version's tree contains a cycle.
- **Citation integrity**: each `citation_key` appears only once per paper in `bibtex`, and every `\cite` key in the text exists in `bibtex`.
- **Cross-version element reuse**: how many elements are shared by more than one version.

**Validation strategy and test cases:** All checks below are computed directly from the output tables, so they run again every time the book is built. The integrity checks end with an `assert`, so the notebook fails loudly if a rule is violated instead of silently reporting bad data.

In [ ]:
ELEMENTS_PATH = PROCESSED_DIR / "elements.parquet"
HIERARCHY_PATH = PROCESSED_DIR / "hierarchy.parquet"
BIBTEX_PATH = PROCESSED_DIR / "bibtex.parquet"
MANIFEST_PATH = PROCESSED_DIR / "parse_manifest.csv"

elements = load_table(ELEMENTS_PATH)
hierarchy = load_table(HIERARCHY_PATH)
bibtex = load_table(BIBTEX_PATH)
parse_manifest = pd.read_csv(MANIFEST_PATH) if MANIFEST_PATH.exists() else pd.DataFrame()

for name, frame in [
    ("elements", elements),
    ("hierarchy", hierarchy),
    ("bibtex", bibtex),
    ("parse_manifest", parse_manifest)
]:
    print(f"{name}: {frame.shape}")
    if not frame.empty:
        display(frame.head())

In [ ]:
results = {}

# Check A: every parent_id must exist as an element_id
if not hierarchy.empty and not elements.empty:
    known_ids = set(elements["element_id"])
    results["edges (total)"] = len(hierarchy)
    results["edges with null parent_id"] = int(hierarchy["parent_id"].isna().sum())
    results["edges whose parent_id is missing"] = int((~hierarchy["parent_id"].isin(known_ids)).sum())
    results["edges whose element_id is missing"] = int((~hierarchy["element_id"].isin(known_ids)).sum())

    # Check B: a version's tree must not contain a cycle
    cyclic = 0
    for _, group in hierarchy.groupby(["arxiv_id", "version"]):
        parent_of = dict(zip(group["element_id"], group["parent_id"]))
        state, has_cycle = {}, False
        for start in parent_of:
            path, node = [], start
            while node in parent_of and state.get(node) != 2:
                if state.get(node) == 1:
                    has_cycle = True
                    break
                state[node] = 1
                path.append(node)
                node = parent_of[node]
            for p in path:
                state[p] = 2
            if has_cycle:
                break
        cyclic += has_cycle
    results["versions containing a cycle"] = cyclic

# Check C: citation_key unique within each paper
if not bibtex.empty:
    dup_rows = bibtex.duplicated(subset=["arxiv_id", "citation_key"], keep=False).sum()
    results["bibtex rows (total)"] = len(bibtex)
    results["bibtex rows with duplicated key in same paper"] = int(dup_rows)

if results:
    display(pd.Series(results, name="value").to_frame())
else:
    print("Tables are empty: integrity checks skipped (no data yet).")

assert results.get("edges with null parent_id", 0) == 0, "Null parent_id found"
assert results.get("edges whose parent_id is missing", 0) == 0, "Some parent_id values do not exist"
assert results.get("edges whose element_id is missing", 0) == 0, "Some element_id values do not exist"
assert results.get("versions containing a cycle", 0) == 0, "A hierarchy contains a cycle"
assert results.get("bibtex rows with duplicated key in same paper", 0) == 0, "Duplicate citation keys"

In [ ]:
if results:
    display(Markdown(
        "**Interpretation.** All three assertions passed, so every `parent_id` resolves to an element, "
        "no version tree has a cycle, and `citation_key` is unique per paper. Orphan parents or cycles would break "
        "tree reconstruction in Milestone 3, and duplicate keys would make `\cite` resolution ambiguous."
    ))
else:
    print("No tables loaded: integrity checks did not run.")

In [ ]:
# Check D: every \cite key in elements must exist in bibtex for the same paper
cite_re = re.compile(r"\\cite[a-zA-Z]*\*?(?:\[[^\]]*\])*\{([^}]*)\}")

if not elements.empty and not bibtex.empty:
    assert set(elements["arxiv_id"]) & set(bibtex["arxiv_id"]), "arxiv_id format differs between elements and bibtex"
    valid_keys = bibtex.groupby("arxiv_id")["citation_key"].apply(set).to_dict()
    cited, unknown = 0, []
    for paper, text in zip(elements["arxiv_id"], elements["content"].fillna("")):
        for group in cite_re.findall(text):
            for key in (k.strip() for k in group.split(",")):
                cited += 1
                if key not in valid_keys.get(paper, set()):
                    unknown.append((paper, key))
    print(f"\cite keys checked: {cited:,}; not found in bibtex: {len(unknown):,}")
    print(unknown[:10])
else:
    print("elements or bibtex is empty: \cite check skipped.")

In [ ]:
# Parsing success rate and coverage (needs the manifest)
if not parse_manifest.empty:
    success_rate = (parse_manifest["status"] == "parsed").mean()
    print(f"Parsing success rate: {success_rate:.1%} of {len(parse_manifest):,} versions")

    non_empty = parse_manifest[parse_manifest["status"] != "empty"]
    print(f"Success rate excluding empty (no-TeX) versions: "
        f"{(non_empty['status'] == 'parsed').mean():.1%} of {len(non_empty):,}")
    
    display(parse_manifest["status"].value_counts().to_frame("versions"))
    print("Versions per paper:")
    display(parse_manifest.groupby("arxiv_id")["version"].nunique().describe().to_frame("versions per paper"))
else:
    print("parse_manifest.csv not found: success rate and empty versions cannot be computed yet.")

# Cross-version element reuse (from hierarchy)
if not hierarchy.empty:
    versions_per_element = hierarchy.groupby(["arxiv_id", "element_id"])["version"].nunique()
    reused = (versions_per_element > 1).sum()
    print(f"Elements shared by 2+ versions: {reused:,} of {len(versions_per_element):,} "
        f"({reused / len(versions_per_element):.1%})")

In [ ]:
# Deduplication: how many elements and bibtex entries were removed by the shared store
if not hierarchy.empty and not elements.empty:
    # "Before": every node occurrence in every version (one hierarchy row = one occurrence)
    elems_before = len(hierarchy)
    # "After": distinct elements in the shared store, excluding document roots
    elems_after = int((elements["kind"] != "document").sum())
    print(f"Elements: {elems_before:,} before -> {elems_after:,} after "
        f"({1 - elems_after / elems_before:.1%} reduction)")

if not bibtex.empty:
    bib_after = len(bibtex)
    if not parse_manifest.empty and "bibtex_raw_count" in parse_manifest.columns:
        # "Before": raw entries counted by the parser per version, summed
        bib_before = int(parse_manifest["bibtex_raw_count"].sum())
        print(f"BibTeX entries: {bib_before:,} before -> {bib_after:,} after "
            f"({1 - bib_after / bib_before:.1%} reduction)")
    else:
        print(f"BibTeX entries after dedup: {bib_after:,} "
            "(before-count unavailable: parse_manifest.csv has no bibtex_raw_count)")

### 7.3 Summary statistics and figures

This section summarizes the processed corpus. Every figure is generated from the output tables (`elements`, `hierarchy`, `bibtex`) and `parse_manifest.csv` each time the book is built, and each one is followed by a short interpretation.

1. **Elements by kind**: distinct elements in the shared store, split into `sentence`, `formula`, `figure`, `heading` and the other kinds.
2. **Deduplication effect**: element rows and BibTeX entries before and after deduplication.
3. **Version coverage and parse status**: versions per paper, and how many versions are parsed, empty (no TeX source) or failed.
4. **Hierarchy size and depth**: number of nodes per version and maximum tree depth per version.

**Caveats.**
- The parse success rate is counted per version, not per paper.
- Empty versions (no TeX source) are kept separate from parser failures, using distinct `status` values in `parse_manifest.csv`.
- The BibTeX "before" count needs the `bibtex_raw_count` column in the manifest.
- Papers with a single version cannot show cross-version deduplication, so the reduction depends on how many papers have several versions.
- Tree depth may be computed on a random sample of versions (cap of x,000, fixed seed); the interpretation under the depth figure states how many versions were used.

In [ ]:
if not elements.empty and "kind" in elements.columns:
    kind_counts = elements["kind"].value_counts(dropna=False)
    ax = kind_counts.plot(kind="bar", figsize=(7, 4))
    ax.set_title("Distinct parsed elements by kind")
    ax.set_xlabel("Element kind")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

    top, share = kind_counts.index[0], kind_counts.iloc[0] / kind_counts.sum()
    display(Markdown(
        f"**Interpretation.** The store holds {kind_counts.sum():,} distinct elements across "
        f"{len(kind_counts)} kinds. The most frequent kind is `{top}` ({share:.1%}). "
        f"The rarest kind is `{kind_counts.index[-1]}` ({kind_counts.iloc[-1]:,} rows)."
    ))
else:
    print("elements table empty: skipped.")

In [ ]:
panels, notes = [], []

if not hierarchy.empty and not elements.empty:
    e_before = len(hierarchy)                              # every node occurrence in every version
    e_after = int((elements["kind"] != "document").sum())  # distinct elements in the shared store
    panels.append(("Elements", ["before", "after"], [e_before, e_after]))
    notes.append(f"Element rows drop from {e_before:,} to {e_after:,} "
                f"({1 - e_after / max(e_before, 1):.1%} reduction), so sentences unchanged across versions are stored once.")

if not bibtex.empty:
    b_after = len(bibtex)
    if not parse_manifest.empty and "bibtex_raw_count" in parse_manifest.columns:
        b_before = int(parse_manifest["bibtex_raw_count"].sum())
        panels.append(("BibTeX entries", ["before", "after"], [b_before, b_after]))
        notes.append(f"BibTeX entries drop from {b_before:,} to {b_after:,} "
                    f"({1 - b_after / max(b_before, 1):.1%} reduction) after merging duplicate keys.")
    else:
        panels.append(("BibTeX entries", ["after"], [b_after]))
        notes.append(f"BibTeX has {b_after:,} entries after dedup; the manifest has no `bibtex_raw_count`, so no before-count.")

if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 4), squeeze=False)
    for ax, (title, labels, values) in zip(axes[0], panels):
        ax.bar(labels, values)
        ax.set_title(title)
        ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()
    display(Markdown("**Interpretation.** " + " ".join(notes)))
else:
    print("elements/hierarchy/bibtex empty: dedup figure skipped.")

In [ ]:
if not parse_manifest.empty:
    per_paper = parse_manifest.groupby("arxiv_id")["version"].nunique()
    status = parse_manifest["status"].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    per_paper.plot(kind="hist", bins=range(1, int(per_paper.max()) + 2), ax=axes[0], align="left")
    axes[0].set_title("Versions per paper")
    axes[0].set_xlabel("Number of versions")
    status.plot(kind="bar", ax=axes[1])
    axes[1].set_title("Version status (parsed / empty / failed)")
    axes[1].set_ylabel("Versions")
    plt.tight_layout()
    plt.show()

    papers_with_empty = parse_manifest.loc[parse_manifest["status"] != "parsed", "arxiv_id"].nunique()
    display(Markdown(
        f"**Interpretation.** {len(per_paper):,} papers have on average {per_paper.mean():.2f} versions "
        f"(max {per_paper.max()}). {status.get('parsed', 0) / len(parse_manifest):.1%} of versions parsed; "
        f"{papers_with_empty:,} papers contain at least one non-parsed (empty or failed) version."
    ))
else:
    print("parse_manifest.csv not found: coverage figure skipped.")

In [ ]:
def max_depth(parent_of):
    """Longest root-to-node path in one version's tree (root = depth 0)."""
    cache = {}
    best = 0
    for start in parent_of:
        path, n = [], start
        while n in parent_of and n not in cache:
            path.append(n)
            n = parent_of[n]
        d = cache.get(n, 0)
        for p in reversed(path):
            d += 1
            cache[p] = d
        best = max(best, cache.get(start, 0))
    return best

if not hierarchy.empty:
    nodes = hierarchy.groupby(["arxiv_id", "version"]).size()

    SAMPLE_N = 2000  # cap on versions used for depth, to keep the book build fast
    keys = nodes.sample(min(SAMPLE_N, len(nodes)), random_state=42).index
    sample = hierarchy.merge(keys.to_frame(index=False), on=["arxiv_id", "version"])

    depths = [max_depth(dict(zip(g["element_id"], g["parent_id"])))
            for _, g in sample.groupby(["arxiv_id", "version"])]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(nodes, bins=40, log=True)
    axes[0].set_title("Nodes per version")
    axes[0].set_xlabel("Nodes")
    axes[1].hist(depths, bins=range(1, max(depths) + 2), align="left")
    axes[1].set_title(f"Max tree depth per version ({len(depths):,} of {len(nodes):,} versions)")
    axes[1].set_xlabel("Depth")
    plt.tight_layout()
    plt.show()

    display(Markdown(
        f"**Interpretation.** Nodes per version: median {int(nodes.median()):,}, "
        f"mean {nodes.mean():,.0f}, max {int(nodes.max()):,}. "
        f"Depth was computed on {len(depths):,} of {len(nodes):,} versions (fixed seed, reproducible): "
        f"median {int(np.median(depths))}, max {max(depths)}."
    ))
else:
    print("hierarchy empty: skipped.")

## 8. Milestone 2 — performance benchmark

Measure on the CPU-only Google Colab testbed:
- End-to-end parser wall time
- Average processing time per publication
- Maximum RAM used
- Final output storage size

| Metric | Value | Measurement method / notes |
|---|---:|---|
| End-to-end parser wall time (s) | TODO | TODO |
| Average time per publication (s) | TODO | TODO |
| Maximum RAM (MiB/GiB) | TODO | TODO |
| Final output size (MiB/GiB) | TODO | TODO |

**Benchmark method:** run through `src/utils/benchmark.py`: wall time with `time.perf_counter`, RAM as RSS of the process and its children sampled every 0.1 s with `psutil` (peak and mean), output size as the total size of the parser output folder.
**Runtime configuration:** CPU-only Google Colab, Python <version>, run date <date>.


In [ ]:
bench_path = PROCESSED_DIR / "benchmarks.csv"
if bench_path.exists():
    run = pd.read_csv(bench_path).query("label == 'm2_parser'").tail(1)   # latest parser run
    if not run.empty:
        r = run.iloc[0]
        display(pd.DataFrame({
            "Metric": ["End-to-end parser wall time (s)", "Average time per publication (s)",
                       "Maximum RAM (MiB)", "Final output size (MiB)"],
            "Value": [r["wall_time_s"], r["time_per_item_s"], r["peak_ram_mib"], r["output_size_mib"]],
        }))
        print("Platform:", r["platform"])
    else:
        print("No m2_parser run in benchmarks.csv yet.")
else:
    print("benchmarks.csv not found: run the benchmark first.")

## 9. Milestone 2 — discussion and limitations

**Difficult LaTeX patterns:** *TODO: list what actually broke, e.g. custom macros, nested environments, `\input` inside macros.*

**Nested sections, lists, formulas, figures and tables:** Nested sections are placed using the level-detection rule of section 6.2, lists are branching components with one child per item, and tables are handled as figures. *TODO: note any case where this rule gave a wrong result.*

**Main causes of parsing failures:** *TODO: use the counts of `failed` versions by cause from `parse_manifest.csv`.*

**Effect of cleanup on deduplication:** *TODO: compare element reduction with and without cleanup, if both were measured. Otherwise write that it was not measured.*

**Bibliography merging and key rewriting:** *TODO: report the result of the citation integrity check from section 7.2.*

**Limitations for Milestone 3:** *TODO: e.g. bibitem fields that could not be recovered, and versions without TeX, since both affect matching.*

**Findings:** TODO

**Limitations and next steps:** TODO


## 10. Overall conclusion

In Milestone 1 the group crawled the assigned arXiv ID ranges, downloaded the `.tex` and `.bib` sources of every version, and collected paper and reference metadata. In Milestone 2 the group built a parser that follows the LaTeX compilation path, builds a hierarchy of sentences, formulas and figures, standardizes the content, converts `\bibitem` entries to BibTeX, and deduplicates references and full-text elements across versions. The result is the `elements`, `hierarchy` and `bibtex` tables, which are the input of the Milestone 3 matching pipeline.

**Key findings:** TODO — parsing success rate, element and BibTeX reduction after deduplication (sections 7.2 and 7.3).

**Performance:** TODO — wall time, peak RAM and output size from section 8.

**Known limitations:** TODO — copy the final list from section 9.


## References

Add complete citations for documentation, libraries, APIs, and external material used in the implementation. At minimum, consult and cite the official project statements linked below.

1. Course Lab Book, *arXiv Project — Problem Statements*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/
2. Course Lab Book, *Project Milestone 1 — Data Crawling*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/
3. Course Lab Book, *Project Milestone 2 — Hierarchical Parsing and Standardization*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/
4. TODO — arXiv API / source access documentation used.
5. TODO — Semantic Scholar API documentation used.
6. TODO — libraries and other external sources used.